# Curación de imágenes: máscara de color de la galaxia central

Limpieza de **contaminación por estrellas** en los cutouts `.jpg` de legacysurvey.org (Fase 2, ver
`README.md`, sección "Limpieza de imágenes"). Objetivo: que en cada imagen solo quede la galaxia
del centro; las estrellas lejos de ella se eliminan, las que caen encima del disco se conservan.

**Idea:** los cutouts vienen centrados en la galaxia objetivo, así que su tono se estima con los
píxeles del centro y el del cielo con el marco de la imagen. Como el estiramiento del visor cambia
los tonos de imagen a imagen, ambos modelos de color se ajustan **por imagen**. La lógica vive en
`src/data/galaxy_mask.py` y los parámetros en `src/config.py` (`GALAXY_MASK_*`); este notebook
solo la explora y la valida visualmente.

Imagen de referencia: `manga-8589-12705.jpg`.

In [ ]:
# importaciones de sistema
import sys
from pathlib import Path

# Referencia a otros archivos sin necesidad de instalar el paquete
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# importaciones de librerias de terceros
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
from scipy import ndimage as ndi

In [ ]:
# configuracion de proyecto
from src.config import (
    CUTOUTS_JPG_DIR,
    CUTOUTS_MASKED_JPG_DIR,
    GALAXY_MASK_BG_PERCENTILE,
    GALAXY_MASK_BORDER_PX,
    GALAXY_MASK_CENTER_FRAC,
    GALAXY_MASK_COV_REG,
    GALAXY_MASK_SMOOTH_SIGMA,
)
from src.data.cutout_io import load_jpg_cutout
from src.data.galaxy_mask import (
    _color_model,
    _mahalanobis_sq,
    apply_mask,
    galaxy_color_mask,
    save_masked_cutout,
)

REFERENCE_NAME = "manga-8589-12705"
names = sorted(p.stem for p in CUTOUTS_JPG_DIR.glob("*.jpg"))
print(f"{len(names)} cutouts en {CUTOUTS_JPG_DIR}")

## 1. Imagen de referencia: regiones de muestreo de color

- **Círculo amarillo**: disco central de radio `GALAXY_MASK_CENTER_FRAC` × lado → tono de la galaxia.
  Con 0.12 abarca bulbo y disco interno; un radio menor solo ve el núcleo (casi saturado,
  amarillento) y el disco azulado de las espirales queda fuera del modelo.
- **Marco cian**: `GALAXY_MASK_BORDER_PX` px de borde → color del cielo (sin su 10% más brillante,
  para no contar las estrellas que caen en el borde).

In [ ]:
ref = load_jpg_cutout(REFERENCE_NAME)
h, w, _ = ref.shape
b = GALAXY_MASK_BORDER_PX

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(ref)
ax.add_patch(Circle(((w - 1) / 2, (h - 1) / 2), GALAXY_MASK_CENTER_FRAC * min(h, w),
                    fill=False, color="yellow", lw=1.5))
ax.add_patch(Rectangle((b / 2, b / 2), w - b, h - b, fill=False, color="cyan", lw=b * 0.35, alpha=0.5))
ax.set_title(f"{REFERENCE_NAME}: regiones de muestreo")
ax.axis("off")
plt.show()

### Colores de galaxia vs. cielo

Distribución RGB (tras el suavizado) de cada región. El núcleo es blanco-amarillento y el disco
azulado, mientras el cielo es oscuro y rojizo: la separación es en **color + brillo**, no solo en
tono. Las estrellas azules tienen un color parecido al disco, por eso el color no basta para
eliminarlas y el paso de **conectividad** (quedarse con la región pegada al centro) es el que las
quita.

In [ ]:
smoothed = ndi.gaussian_filter(ref.astype(float), sigma=(GALAXY_MASK_SMOOTH_SIGMA, GALAXY_MASK_SMOOTH_SIGMA, 0))
yy, xx = np.mgrid[:h, :w]
radius = np.hypot(yy - (h - 1) / 2, xx - (w - 1) / 2)
border = np.zeros((h, w), bool)
border[:b] = border[-b:] = True
border[:, :b] = border[:, -b:] = True

sky = smoothed[border]
sky = sky[sky.mean(1) < np.percentile(sky.mean(1), GALAXY_MASK_BG_PERCENTILE)]
regions = {
    "galaxia (centro)": smoothed[radius <= GALAXY_MASK_CENTER_FRAC * min(h, w)],
    "cielo (marco)": sky,
}
display(pd.DataFrame(
    {k: np.median(v, axis=0).round(1) for k, v in regions.items()}, index=["R", "G", "B"]
).T)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for (label, px), color in zip(regions.items(), ["tab:orange", "tab:blue"]):
    axes[0].scatter(px[:, 0], px[:, 2], s=2, alpha=0.3, label=label, color=color)
    axes[1].scatter(px.mean(1), px[:, 2] - px[:, 0], s=2, alpha=0.3, label=label, color=color)
axes[0].set(xlabel="R", ylabel="B", title="R vs. B")
axes[1].set(xlabel="brillo medio", ylabel="B - R (azul > 0)", title="Brillo vs. color")
for ax in axes:
    ax.legend(markerscale=5)
plt.tight_layout()
plt.show()

## 2. Paso a paso sobre la referencia

1. Candidatos: píxeles cuyo color está más cerca (distancia de Mahalanobis) del modelo de la
   galaxia que del modelo del cielo. Aquí aparecen también las estrellas.
2. Máscara final: apertura morfológica (corta puentes finos), componente conexa que contiene el
   centro, cierre y relleno de huecos (`src/data/galaxy_mask.py`).

In [ ]:
candidates = (
    _mahalanobis_sq(smoothed, _color_model(regions["galaxia (centro)"], GALAXY_MASK_COV_REG))
    < _mahalanobis_sq(smoothed, _color_model(regions["cielo (marco)"], GALAXY_MASK_COV_REG))
)
mask = galaxy_color_mask(ref)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
panels = [(ref, "original"), (candidates, "1. candidatos por color"),
          (mask, "2. componente central"), (apply_mask(ref, mask), "resultado")]
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"Fracción de la imagen cubierta por la galaxia: {mask.mean():.1%}")

## 3. Validación visual sobre toda la muestra

Original / enmascarada, lado a lado (encima de la enmascarada, % de la imagen que cubre la
máscara). Revisar sobre todo: (a) estrellas lejanas eliminadas, (b) disco externo débil
conservado, (c) galaxias vecinas.

In [ ]:
coverage = {}
n_cols = 5
n_rows = int(np.ceil(len(names) / n_cols))
fig, axes = plt.subplots(n_rows, 2 * n_cols, figsize=(2 * n_cols * 1.8, n_rows * 1.9))
for ax in axes.ravel():
    ax.axis("off")
for i, name in enumerate(names):
    image = load_jpg_cutout(name)
    m = galaxy_color_mask(image)
    coverage[name] = m.mean()
    r, c = divmod(i, n_cols)
    axes[r, 2 * c].imshow(image)
    axes[r, 2 * c].set_title(name.replace("manga-", ""), fontsize=7)
    axes[r, 2 * c + 1].imshow(apply_mask(image, m))
    axes[r, 2 * c + 1].set_title(f"{m.mean():.0%}", fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
coverage = pd.Series(coverage, name="fraccion_mascara").sort_values()
display(coverage.describe().to_frame().T)
print("Menor cobertura (posible máscara recortada de más):")
display(coverage.head(5).to_frame())
print("Mayor cobertura (posible fusión con vecinas o halo de fondo):")
display(coverage.tail(5).to_frame())

### Observaciones

- Las estrellas separadas de la galaxia se eliminan en prácticamente toda la muestra, incluida la
  estrella azul brillante de la imagen de referencia.
- Las estrellas superpuestas al disco se conservan (no se pueden separar por conectividad), como
  se acordó.
- **Limitación:** si una galaxia vecina se superpone a la central (pares en interacción, grupos),
  queda dentro de la misma máscara. Separarlas requeriría deblending (p. ej. watershed sobre el
  brillo o `photutils.segmentation.deblend_sources`), fuera del alcance de esta máscara de color.
- **Limitación:** en algunas espirales con bulbo muy dominante, el disco externo débil tiene un
  color más cercano al cielo que al centro y queda fuera de la máscara.

## 4. Guardar los cutouts enmascarados

Escribe cada imagen con el fondo en negro en `CUTOUTS_MASKED_JPG_DIR`
(`data/legacy_cutouts/jpg_masked/`, no versionado).

In [ ]:
saved = {name: save_masked_cutout(name) for name in names}
empty = [n for n, f in saved.items() if f == 0]
print(f"{len(saved)} imágenes guardadas en {CUTOUTS_MASKED_JPG_DIR}; máscaras vacías: {empty}")